In [5]:
import pandas as pd
from sqlalchemy import create_engine
import urllib.parse
from dotenv import load_dotenv
import os

In [6]:
load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS")
DB_HOST = os.getenv("DB_HOST")
DB_NAME = os.getenv("DB_NAME")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:5432/{DB_NAME}"
)

In [11]:
df = pd.read_sql("SELECT * FROM customer_behavior_raw", engine)

print("Shape:", df.shape)
df.head()


Shape: (1000000, 60)


,user_id,age,gender,country,urban_rural,income_level,employment_status,education_level,relationship_status,has_children,...,cart_items_average,checkout_abandonments_per_month,purchase_conversion_rate,app_usage_frequency,notification_response_rate,account_age_months,last_purchase_date,social_sharing_frequency,premium_subscription,return_rate
0,1,56,Female,Germany,Suburban,90860,Self-employed,Associate Degree,Single,0,...,10,2,62,7,74,19,2025-06-22,6,1,50
1,2,69,Male,Japan,Suburban,35423,Unemployed,Bachelor,Single,1,...,5,7,54,5,23,8,2026-07-25,3,0,37
2,3,46,Female,India,Urban,21467,Self-employed,Associate Degree,Married,1,...,3,3,33,7,12,13,2026-02-26,6,0,53
3,4,32,Male,Canada,Urban,41770,Self-employed,Bachelor,Widowed,0,...,5,9,26,4,19,9,2026-10-27,7,0,98
4,5,60,Female,Japan,Urban,183882,Employed,Associate Degree,Widowed,1,...,8,0,18,7,30,3,2026-06-23,3,0,86


In [21]:

import numpy as np

impulse_median = df["impulse_buying_score"].median()

conditions = [
    df["impulse_buying_score"] > impulse_median * 1.2,
    df["impulse_buying_score"] > impulse_median,
    df["impulse_buying_score"] <= impulse_median
]

choices = [
    "loyal_customers",
    "potential_loyalists",
    "at_risk_customers"
]

df["customer_segment"] = np.select(conditions, choices,default="general_customers")

In [6]:
import numpy as np

In [22]:

impulse_median = df["impulse_buying_score"].median()

conditions = [
    
    (df["impulse_buying_score"] > impulse_median *1.2),

     
    (df["impulse_buying_score"] > impulse_median),

    
    (df["impulse_buying_score"] <= impulse_median)
]

choices = [
    "loyal_customers",
    "potential_loyalists",
    "at_risk_customers"
]
df["customer_segment"] = np.select(conditions, choices,default="general_customers")



In [25]:
df["customer_segment"].value_counts()

customer_segment
at_risk_customers      545613
loyal_customers        363878
potential_loyalists     90509
Name: count, dtype: int64

In [26]:
df["customer_segment"].value_counts(normalize=True) * 100

customer_segment
at_risk_customers      54.5613
loyal_customers        36.3878
potential_loyalists     9.0509
Name: proportion, dtype: float64

In [30]:

recommendation_map = {
    "loyal_customers": "Premium product recommendations",
    "potential_loyalists": "Personalized bundle offers",
    "at_risk_customers": "Discount and retention offers",
}

df["recommendation_strategy"] = df["customer_segment"].map(recommendation_map)

In [31]:
df[["customer_segment", "recommendation_strategy"]].head()

,customer_segment,recommendation_strategy
0,at_risk_customers,Discount and retention offers
1,at_risk_customers,Discount and retention offers
2,at_risk_customers,Discount and retention offers
3,at_risk_customers,Discount and retention offers
4,loyal_customers,Premium product recommendations


In [32]:
df.groupby("customer_segment")["impulse_buying_score"].mean()

customer_segment
at_risk_customers      2.463112
loyal_customers        8.550841
potential_loyalists    6.000000
Name: impulse_buying_score, dtype: float64

In [37]:
df_sample = df.sample(100000)
df_sample.to_sql("customer_behavior_final", engine, if_exists="replace", index=False)

397

In [38]:
pd.read_sql("SELECT COUNT(*) FROM customer_behavior_final", engine)


,count
0,100000


In [39]:
df_final = pd.read_sql("SELECT * FROM customer_behavior_final", engine)
df_final.head()


,user_id,age,gender,country,urban_rural,income_level,employment_status,education_level,relationship_status,has_children,...,purchase_conversion_rate,app_usage_frequency,notification_response_rate,account_age_months,last_purchase_date,social_sharing_frequency,premium_subscription,return_rate,customer_segment,recommendation_strategy
0,990647,33,Female,France,Urban,143284,Self-employed,Bachelor,Widowed,0,...,55,4,36,17,2025-07-24,7,0,6,at_risk_customers,Discount and retention offers
1,636125,70,Male,Brazil,Rural,164785,Retired,Master,Married,1,...,1,5,11,5,2026-08-23,1,1,65,loyal_customers,Premium product recommendations
2,418953,20,Male,China,Urban,179939,Retired,Master,Widowed,1,...,90,0,13,10,2026-01-09,4,1,8,loyal_customers,Premium product recommendations
3,96762,22,Male,UK,Suburban,57384,Unemployed,Bachelor,Single,0,...,10,7,20,9,2026-09-02,7,1,38,at_risk_customers,Discount and retention offers
4,209093,41,Male,India,Urban,64232,Retired,Bachelor,In a relationship,1,...,21,5,78,17,2025-12-21,6,1,85,at_risk_customers,Discount and retention offers
